In [ ]:
# ============================================================
# LAB 03 - EXPANSÃO DE CLASSE
# DATA DRIFT & NOVAS INTENÇÕES
# SAC IMOBILIÁRIO - NLU
# ============================================================
#
# Alterações deste laboratório:
#
# 1 - Nova intenção: cancelar_contrato
# 2 - 5 novas frases de treinamento
# 3 - Nova resposta no RESPOSTAS_PADRAO
# 4 - Modelo treinado novamente com 5 classes
# 5 - Limiar de confiança mantido em 65%
# 6 - Teste da nova intenção no Gradio
#
# ============================================================


# ============================================================
# BLOCO 1 - PREPARAÇÃO DO AMBIENTE
# ============================================================

!pip install -q gensim gradio
!python -m spacy download pt_core_news_sm -q


# ============================================================
# IMPORTAÇÕES
# ============================================================

import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api

from sklearn.linear_model import LogisticRegression

import gradio as gr


# ============================================================
# 1. CARREGAMENTO DO SPACY
# ============================================================

print("Carregando modelo morfológico Spacy...")

nlp = spacy.load("pt_core_news_sm")


# ============================================================
# 2. CARREGAMENTO DOS WORD EMBEDDINGS
# ============================================================

print("Carregando espaço vetorial denso de embeddings GloVe...")

word_vectors = api.load("glove-wiki-gigaword-50")

print("Modelos carregados!")


# ============================================================
# 3. DATASET SUPERVISIONADO
# ============================================================

dados_imobiliaria = [

    # --------------------------------------------------------
    # INTENÇÃO 1 - COMPRAR IMÓVEL
    # --------------------------------------------------------

    ("Quero comprar um apartamento de 3 quartos com varanda",
     "comprar_imovel"),

    ("Gostaria de ver casas à venda no centro da cidade",
     "comprar_imovel"),

    ("Qual o preço médio para compra de cobertura com piscina?",
     "comprar_imovel"),

    ("Procuro imóvel residencial para comprar com financiamento",
     "comprar_imovel"),

    ("Vocês têm sobrados à venda na zona sul?",
     "comprar_imovel"),


    # --------------------------------------------------------
    # INTENÇÃO 2 - ALUGAR IMÓVEL
    # --------------------------------------------------------

    ("Procurando kitnet para alugar perto da faculdade",
     "alugar_imovel"),

    ("Qual o valor do aluguel deste apartamento de 2 dormitórios?",
     "alugar_imovel"),

    ("Quero alugar um galpão comercial para minha empresa",
     "alugar_imovel"),

    ("Quais imóveis estão disponíveis para locação imediata?",
     "alugar_imovel"),

    ("Preciso de uma casa para alugar que aceite animais",
     "alugar_imovel"),


    # --------------------------------------------------------
    # INTENÇÃO 3 - SUPORTE E MANUTENÇÃO
    # --------------------------------------------------------

    ("O chuveiro do apartamento alugado queimou, como pedir conserto?",
     "suporte_manutencao"),

    ("Muro da casa está com infiltração e vazamento de água",
     "suporte_manutencao"),

    ("Preciso do contato do encanador para reparo na cozinha",
     "suporte_manutencao"),

    ("A porta da varanda quebrou, quem faz a manutenção?",
     "suporte_manutencao"),

    ("Vazamento no teto do banheiro precisa de reparo urgente",
     "suporte_manutencao"),


    # --------------------------------------------------------
    # INTENÇÃO 4 - SEGUNDA VIA / BOLETO / CONTRATO
    # --------------------------------------------------------

    ("Como faço para baixar a segunda via do boleto do aluguel?",
     "2via_boleto_contrato"),

    ("Não recebi o boleto deste mês para pagamento",
     "2via_boleto_contrato"),

    ("Preciso do informe de rendimentos e cópia do contrato",
     "2via_boleto_contrato"),

    ("Onde pego o boleto atualizado com o valor do condomínio?",
     "2via_boleto_contrato"),

    ("Quero solicitar a segunda via do recibo de pagamento",
     "2via_boleto_contrato"),


    # --------------------------------------------------------
    # INTENÇÃO 5 - CANCELAR CONTRATO
    # --------------------------------------------------------
    #
    # NOVA CLASSE DO LAB 03
    #

    ("Quero cancelar meu contrato de aluguel",
     "cancelar_contrato"),

    ("Como faço para cancelar o contrato do imóvel?",
     "cancelar_contrato"),

    ("Gostaria de solicitar o distrato do contrato",
     "cancelar_contrato"),

    ("Preciso encerrar meu contrato de aluguel",
     "cancelar_contrato"),

    ("Quero saber como funciona o cancelamento do contrato",
     "cancelar_contrato")
]


# ============================================================
# CRIAÇÃO DO DATAFRAME
# ============================================================

df = pd.DataFrame(
    dados_imobiliaria,
    columns=["mensagem", "intencao"]
)


print()
print("============================================================")
print("DATASET")
print("============================================================")

print(
    f"Dataset carregado com {len(df)} mensagens "
    f"divididas em {df['intencao'].nunique()} intenções."
)

print()
print("Quantidade por intenção:")
print(df["intencao"].value_counts())


# ============================================================
# BLOCO 2 - ESTEIRA NLU
# PRÉ-PROCESSAMENTO E VECTORIZATION
# ============================================================


# ============================================================
# FUNÇÃO DE PRÉ-PROCESSAMENTO
# ============================================================

def preprocessar_texto(texto: str) -> str:

    """
    Realiza a limpeza e padronização do texto:

    1. Converte para minúsculas
    2. Remove números e caracteres especiais
    3. Remove stop words
    4. Extrai os lemas usando Spacy
    """

    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens = [

        token.lemma_

        for token in doc

        if not token.is_stop
        and not token.is_space
        and len(token.text) > 1

    ]

    return " ".join(tokens)


# ============================================================
# FUNÇÃO DE SENTENCE EMBEDDING
# MEAN POOLING
# ============================================================

def extrair_sentence_embedding(
    texto_limpo: str,
    modelo_emb
) -> np.ndarray:

    """
    Converte uma frase em um vetor denso
    utilizando Mean Pooling.
    """

    palavras = texto_limpo.split()

    vetores = [

        modelo_emb[p]

        for p in palavras

        if p in modelo_emb

    ]

    # Segurança para frases sem palavras conhecidas
    if len(vetores) == 0:

        return np.zeros(
            modelo_emb.vector_size
        )

    # Média dos vetores
    return np.mean(
        vetores,
        axis=0
    )


# ============================================================
# APLICAÇÃO DO PRÉ-PROCESSAMENTO
# ============================================================

df["mensagem_limpa"] = df["mensagem"].apply(
    preprocessar_texto
)


# ============================================================
# CRIAÇÃO DA MATRIZ X
# ============================================================

X_densos = np.array([

    extrair_sentence_embedding(
        texto,
        word_vectors
    )

    for texto in df["mensagem_limpa"]

])


# ============================================================
# CRIAÇÃO DO ALVO Y
# ============================================================

y = df["intencao"].values


print()
print("============================================================")
print("VECTORIZATION")
print("============================================================")

print(
    "Formato de X:",
    X_densos.shape
)

print(
    "Quantidade de rótulos:",
    len(y)
)


# ============================================================
# BLOCO 3 - TREINAMENTO DO MODELO
# ============================================================


print()
print("============================================================")
print("TREINAMENTO DO MODELO")
print("============================================================")


# ============================================================
# REGRESSÃO LOGÍSTICA
# ============================================================

modelo_nlu = LogisticRegression(
    C=1.0,
    max_iter=500
)


# ============================================================
# TREINAMENTO
# ============================================================

modelo_nlu.fit(
    X_densos,
    y
)


print("Modelo supervisionado treinado!")


# ============================================================
# VERIFICAR AS CLASSES APRENDIDAS
# ============================================================

print()
print("Classes aprendidas pelo modelo:")

for classe in modelo_nlu.classes_:

    print("-", classe)


# ============================================================
# RESPOSTAS PADRÃO DO SAC
# ============================================================

RESPOSTAS_PADRAO = {

    # --------------------------------------------------------
    # COMPRA
    # --------------------------------------------------------

    "comprar_imovel": (

        "**Atendimento de Vendas:** "
        "Ficamos felizes com seu interesse! "

        "Você pode conferir nosso catálogo de imóveis "
        "à venda em nosso site "
        "www.imobiliaria.com/vendas "

        "ou aguardar que um de nossos corretores "
        "entrará em contato em instantes."

    ),


    # --------------------------------------------------------
    # ALUGUEL
    # --------------------------------------------------------

    "alugar_imovel": (

        "**Atendimento de Locação:** "
        "Temos ótimas opções disponíveis! "

        "Acesse "
        "www.imobiliaria.com/aluguel "

        "para filtrar por região e valor. "

        "Para agendar uma visita, envie o código "
        "do imóvel por aqui."

    ),


    # --------------------------------------------------------
    # SUPORTE
    # --------------------------------------------------------

    "suporte_manutencao": (

        "**Suporte e Manutenção:** "
        "Sentimos muito pelo inconveniente. "

        "Por favor, abra um chamado urgente "
        "em nosso portal do inquilino "

        "(www.imobiliaria.com/manutencao) "

        "anexando fotos ou vídeos do problema "
        "para acionarmos nossos prestadores."

    ),


    # --------------------------------------------------------
    # BOLETO / CONTRATO
    # --------------------------------------------------------

    "2via_boleto_contrato": (

        "**Financeiro e Contratos:** "
        "Para acessar boletos ou documentos, "

        "acesse a Área do Cliente em "
        "www.imobiliaria.com/cliente "

        "informando seu CPF e senha. "

        "Lá você baixa a 2ª via atualizada "
        "em segundos."

    ),


    # --------------------------------------------------------
    # NOVA INTENÇÃO - CANCELAMENTO
    # --------------------------------------------------------

    "cancelar_contrato": (

        "**Atendimento de Contratos:** "

        "Para solicitar o cancelamento ou distrato "
        "do contrato, entre em contato com nosso "
        "setor de contratos para receber as orientações "

        "e verificar as condições previstas "
        "no contrato."

    )
}


# ============================================================
# BLOCO 4 - MOTOR DE INFERÊNCIA
# COM FALLBACK DINÂMICO
# ============================================================


# ============================================================
# LIMIAR DE CONFIANÇA
# ============================================================

# Mantido em 65% conforme o laboratório anterior

LIMIAR_CONFIANCA = 0.65


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def processar_atendimento_sac(
    mensagem_usuario: str
):

    # --------------------------------------------------------
    # VERIFICAÇÃO DE ENTRADA
    # --------------------------------------------------------

    if (
        not mensagem_usuario
        or not mensagem_usuario.strip()
    ):

        return (
            "N/A",
            "0.0%",
            "Aguardando mensagem...",
            "Aguardando entrada do usuário..."
        )


    # --------------------------------------------------------
    # STEP 1 - PRÉ-PROCESSAMENTO
    # --------------------------------------------------------

    msg_limpa = preprocessar_texto(
        mensagem_usuario
    )


    # --------------------------------------------------------
    # STEP 2 - VECTORIZATION
    # --------------------------------------------------------

    vetor_input = (

        extrair_sentence_embedding(
            msg_limpa,
            word_vectors
        )

        .reshape(1, -1)

    )


    # --------------------------------------------------------
    # STEP 3 - PREDIÇÃO
    # --------------------------------------------------------

    probabilidades = (

        modelo_nlu
        .predict_proba(vetor_input)[0]

    )


    # --------------------------------------------------------
    # MAIOR PROBABILIDADE
    # --------------------------------------------------------

    idx_maior_prob = np.argmax(
        probabilidades
    )


    confianca = probabilidades[
        idx_maior_prob
    ]


    intencao_detectada = (
        modelo_nlu.classes_[
            idx_maior_prob
        ]
    )


    percentual_confianca = (
        f"{confianca * 100:.1f}%"
    )


    # --------------------------------------------------------
    # STEP 4 - REGRA DE DECISÃO
    # --------------------------------------------------------

    if confianca >= LIMIAR_CONFIANCA:

        classificacao_status = (

            f"IDENTIFICADO "
            f"({intencao_detectada}) "

            f"| Confiança: "
            f"{percentual_confianca} "

            f"| Corte mínimo: 65%"

        )

        texto_resposta = (
            RESPOSTAS_PADRAO[
                intencao_detectada
            ]
        )


    else:

        classificacao_status = (

            f"UNCERTAIN "
            f"(Fallback Acionado) "

            f"| Confiança: "
            f"{percentual_confianca} "

            f"| Corte mínimo: 65%"

        )

        texto_resposta = (

            "Desculpe, não consegui compreender "
            "com clareza a sua solicitação. "

            "Estou transferindo agora mesmo sua "
            "conversa para um de nossos atendentes. "

            "Por favor, aguarde um momento."

        )


    # --------------------------------------------------------
    # CARD DA RESPOSTA
    # --------------------------------------------------------

    card_resposta = f"""

    <div style="
        background-color: #f0f4f9;
        border-left: 5px solid #2b5c8f;
        padding: 15px;
        border-radius: 8px;
        margin-top: 10px;
    ">

        <h4 style="
            margin: 0 0 8px 0;
            color: #2b5c8f;
        ">

            Resposta Automática do SAC:

        </h4>

        <p style="
            margin: 0;
            font-size: 15px;
            color: #1a1a1a;
        ">

            {texto_resposta}

        </p>

    </div>

    """


    # --------------------------------------------------------
    # RETORNO
    # --------------------------------------------------------

    return (

        intencao_detectada,
        percentual_confianca,
        classificacao_status,
        card_resposta

    )


# ============================================================
# TESTE AUTOMÁTICO DA NOVA CLASSE
# ============================================================

print()
print("============================================================")
print("TESTE DA NOVA INTENÇÃO")
print("============================================================")


resultado_teste = processar_atendimento_sac(
    "Quero cancelar meu contrato de aluguel"
)


print(
    "Intenção:",
    resultado_teste[0]
)

print(
    "Confiança:",
    resultado_teste[1]
)

print(
    "Status:",
    resultado_teste[2]
)


# ============================================================
# BLOCO 5 - INTERFACE GRADIO
# ============================================================


print()
print("============================================================")
print("CRIANDO INTERFACE GRADIO")
print("============================================================")


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SAC Imobiliário"
) as app:


    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    gr.Markdown(
        """
        # 🏠 SAC Imobiliário — ChatBot Inteligente

        *Será um prazer atendê-lo.
        Digite a seguir a sua necessidade.*
        """
    )


    with gr.Row():


        # ====================================================
        # COLUNA ESQUERDA
        # ====================================================

        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "### 📝 Mensagem do Cliente"
            )


            input_texto = gr.Textbox(

                lines=4,

                placeholder=(
                    "Ex: Quero cancelar meu contrato..."
                ),

                label="Digite sua necessidade"

            )


            btn_processar = gr.Button(

                "🔵 Processar Mensagem",

                variant="primary",

                size="lg"

            )


            # ------------------------------------------------
            # EXEMPLOS
            # ------------------------------------------------

            gr.Examples(

                examples=[

                    [
                        "Preciso de suporte técnico "
                        "para consertar vazamento."
                    ],

                    [
                        "Quero ver apartamentos "
                        "à venda na zona sul."
                    ],

                    [
                        "Como faço para alugar "
                        "um galpão comercial?"
                    ],

                    [
                        "Gostaria de baixar "
                        "o boleto do condomínio."
                    ],

                    [
                        "Quero cancelar meu contrato "
                        "de aluguel."
                    ],

                    [
                        "Gostaria de solicitar "
                        "o distrato do contrato."
                    ],

                    [
                        "Vocês vendem terreno "
                        "na Lua ou em Marte?"
                    ]

                ],

                inputs=input_texto

            )


        # ====================================================
        # COLUNA DIREITA
        # ====================================================

        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "### 📊 Painel de Diagnóstico"
            )


            # ------------------------------------------------
            # INDICADORES
            # ------------------------------------------------

            with gr.Row():


                out_intencao = gr.Textbox(

                    label="Intenção",

                    scale=2,

                    interactive=False

                )


                out_confianca = gr.Textbox(

                    label="Confiança",

                    scale=1,

                    interactive=False

                )


            # ------------------------------------------------
            # STATUS
            # ------------------------------------------------

            out_status = gr.Textbox(

                label="Status da Decisão",

                interactive=False

            )


            # ------------------------------------------------
            # RESPOSTA
            # ------------------------------------------------

            out_resposta = gr.HTML(

                value=(
                    "<div style="
                    "'padding: 15px; color: #888;'>"
                    "Aguardando envio de mensagem..."
                    "</div>"
                ),

                label="Resposta da Imobiliária"

            )


    # ========================================================
    # AÇÃO DO BOTÃO
    # ========================================================

    btn_processar.click(

        fn=processar_atendimento_sac,

        inputs=[
            input_texto
        ],

        outputs=[
            out_intencao,
            out_confianca,
            out_status,
            out_resposta
        ]

    )


# ============================================================
# EXECUTAR A APLICAÇÃO
# ============================================================

print()
print("============================================================")
print("INICIANDO GRADIO")
print("============================================================")

print(
    "Aguarde o link do Gradio aparecer..."
)

app.launch(
    debug=True,
    share=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 78.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Carregando modelo morfológico Spacy...
Carregando espaço vetorial denso de embeddings GloVe...
Modelos carregados!

DATASET
Dataset carregado com 25 mensagens divididas em 5 intenções.

Quantidade por intenção:
intencao
comprar_imovel          5
alugar_imovel           5
suporte_manutencao      5
2via_boleto_contrato    5
cancelar_contrato       5
Name: count, dtype: int64

VECTORIZATION
Formato de X: (25, 50)
Quantidade de rótulos: 25

TREINAMENTO DO MODELO
Modelo supervisionado treinado!

Classes aprendidas pelo modelo:
- 2via_boleto_contrato
- alugar_imovel
- cancelar_cont

/tmp/ipykernel_2257/1684989626.py:731: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(



INICIANDO GRADIO
Aguarde o link do Gradio aparecer...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95936990d7cd5ccbc3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
